<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
# НОВЫЙ ПУТЬ: Добавляем путь к данным взаимодействий за нужный период
TRACKER_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_tracker_data/final_apparel_tracker_data_08'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [2]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [3]:
# Шаг 2.5. Загрузка взаимодействий (tracker)
def load_tracker():
    print("Загружаем тренировочные данные взаимодействий...")
    tracker_data = []
    # Используем rglob для рекурсивного поиска всех .parquet файлов
    # в директории TRACKER_PATH и её поддиректориях
    for f in Path(TRACKER_PATH).rglob('*.parquet'):
        tracker_data.append(pd.read_parquet(f))

    if tracker_data:
        df = pd.concat(tracker_data, ignore_index=True)
        print(f"Загружено взаимодействий: {len(df):,}")
        return df
    else:
        print("Файлы взаимодействий не найдены.")
        return pd.DataFrame() # Возвращаем пустой DataFrame

tracker_df = load_tracker()

Загружаем тренировочные данные взаимодействий...
Загружено взаимодействий: 116,552,409


In [4]:
print("\n\n=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(tracker_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in tracker_df.columns:
    print(f"  • {col}: {tracker_df[col].dtype}")



=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------------+-----------+-----------+---------------------+---------------+
|    | action_widget   |   item_id |   user_id | timestamp           | action_type   |
+====+=================+===========+===========+=====================+===============+
|  0 | pdp             |    996252 |   3542470 | 2025-07-08 23:29:23 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+
|  1 | pdp             |   4127285 |   3267450 | 2025-07-09 14:51:02 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+

📊 Схема данных:
  • action_widget: object
  • item_id: int32
  • user_id: int32
  • timestamp: datetime64[ns]
  • action_type: object


In [5]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:00<00:00,  4.78it/s]

Найдено уникальных тестовых пользователей: 470,347


In [6]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [7]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [8]:
print("\n\nАНАЛИЗ ВЗАИМОДЕЙСТВИЙ")
print("=" * 50)
print(f"Общее количество взаимодействий: {len(tracker_df):,}")
print(f"Уникальных пользователей: {tracker_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {tracker_df['item_id'].nunique():,}")

if 'timestamp' in tracker_df.columns:
    min_date = tracker_df['timestamp'].min().date()
    max_date = tracker_df['timestamp'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'action_type' in tracker_df.columns:
    print("\nРаспределение типов действий:")
    action_counts = tracker_df['action_type'].value_counts()
    action_counts_pct = tracker_df['action_type'].value_counts(normalize=True) * 100
    for action, count in action_counts.items():
        pct = action_counts_pct[action]
        print(f"  {action}: {count:,} ({pct:.1f}%)")



АНАЛИЗ ВЗАИМОДЕЙСТВИЙ
Общее количество взаимодействий: 116,552,409
Уникальных пользователей: 807,670
Уникальных товаров: 3,197,923
Период данных: 2010-01-30 - 2025-07-16

Распределение типов действий:
  page_view: 80,605,340 (69.2%)
  view_description: 17,253,961 (14.8%)
  review_view: 5,729,054 (4.9%)
  to_cart: 4,618,070 (4.0%)
  favorite: 3,455,017 (3.0%)
  remove: 3,007,598 (2.6%)
  unfavorite: 1,883,369 (1.6%)


In [9]:
# Для tracker_df по timestamp
if 'timestamp' in tracker_df.columns:
    print("\nКоличество взаимодействий по годам (на основе timestamp):")
    tracker_by_year = tracker_df['timestamp'].dt.year.value_counts().sort_index()
    for year, count in tracker_by_year.items():
        print(f"  {year}: {count:,}")


Количество взаимодействий по годам (на основе timestamp):
  2010: 1
  2024: 12
  2025: 116,552,396


In [10]:
print("\n\nАНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)")
print("=" * 50)
# Объединим item_id из orders и tracker для более полной картины
all_item_ids = set(orders_df['item_id'].unique()).union(set(tracker_df['item_id'].unique()))
print(f"Общее количество уникальных товаров в заказах и взаимодействиях: {len(all_item_ids):,}")



АНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)
Общее количество уникальных товаров в заказах и взаимодействиях: 4,797,217


In [11]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов


In [12]:
# === Адаптированная функция 1: build_popularity_model (с tracker_df, обработка батчами внутри памяти) ===
def build_popularity_model(orders_df, tracker_df, period_start='2025-07-02', period_end='2025-07-15', top_k=100, chunk_size=5_000_000):
    """
    Строит список популярных товаров на основе комбинированного рейтинга
    (покупки + просмотры) в указанный период.
    Обрабатывает tracker_df по частям для экономии памяти.
    """
    import pandas as pd
    from collections import defaultdict
    import gc # Для ручной очистки памяти

    print("Строим модель популярности (с учетом просмотров, батчами)...")

    # --- 1. Обработка заказов (orders_df) ---
    popular_orders = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['created_date'] >= pd.to_datetime(period_start)) &
        (orders_df['created_date'] <= pd.to_datetime(period_end))
    ].copy() # copy() для избежания SettingWithCopyWarning, если нужно
    print(f"  Отфильтровано заказов за период {period_start} - {period_end}: {len(popular_orders):,}")
    item_purchases = popular_orders['item_id'].value_counts().to_dict()
    print(f"  Уникальных купленных товаров в периоде: {len(item_purchases):,}")
    # Освобождаем память от временного фрейма
    del popular_orders
    gc.collect()

    # --- 2. Обработка просмотров (tracker_df) по частям ---
    print(f"  Обрабатываем tracker_df по частям размером ~{chunk_size:,} записей...")
    item_views = defaultdict(int) # Используем defaultdict для накопления счетчиков

    # Создаем итератор по частям tracker_df
    # np.array_split не подходит, так как tracker_df уже в памяти и большой
    # Лучше использовать iloc для нарезки
    total_rows = len(tracker_df)
    num_chunks = (total_rows // chunk_size) + 1

    for i in range(num_chunks):
        start_row = i * chunk_size
        end_row = min((i + 1) * chunk_size, total_rows)
        chunk = tracker_df.iloc[start_row:end_row]

        # Фильтрация чанка
        filtered_chunk = chunk[
            (chunk['action_type'] == 'page_view') &
            (chunk['timestamp'] >= pd.to_datetime(period_start)) &
            (chunk['timestamp'] <= pd.to_datetime(period_end))
        ]

        # Агрегация
        chunk_counts = filtered_chunk['item_id'].value_counts()
        for item_id, count in chunk_counts.items():
            item_views[item_id] += count

        # Очистка ссылок на чанк
        del chunk, filtered_chunk, chunk_counts
        if i % 10 == 0 or i == num_chunks - 1: # Промежуточный лог
             print(f"    Обработано {i+1}/{num_chunks} частей tracker_df")
        gc.collect() # Принудительная сборка мусора

    print(f"  Всего уникальных просмотренных товаров в периоде: {len(item_views):,}")

    # --- 3. Комбинирование рейтингов ---
    combined_scores = defaultdict(float)
    purchase_weight = 3.0
    view_weight = 1.0

    print("  Рассчитываем комбинированный рейтинг...")
    for item_id, count in item_purchases.items():
        combined_scores[item_id] += count * purchase_weight

    for item_id, count in item_views.items():
        combined_scores[item_id] += count * view_weight

    print(f"  Уникальных товаров с комбинированным рейтингом: {len(combined_scores):,}")

    # --- 4. Сортировка и выбор топ-K ---
    sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    top_items = [item_id for item_id, score in sorted_items[:top_k]]
    print(f"  Выбрано топ-{len(top_items)} популярных товаров (покупки+просмотры)")

    # --- 5. Вывод топ-10 для анализа ---
    print("\nТоп-10 самых популярных товаров (покупки*3 + просмотры*1):")
    for i, (item_id, score) in enumerate(sorted_items[:10], 1):
         print(f"  {i}. Товар {item_id}: {score:.1f} баллов")

    # Очистка
    del item_purchases, item_views, combined_scores, sorted_items
    gc.collect()

    return top_items


In [14]:
# 1. Построение популярности
# Передаем tracker_df, функция сама разобьет его на части
popular_items = build_popularity_model(
    orders_df,
    tracker_df,
    period_start='2025-07-02',
    period_end='2025-07-15',
    top_k=100,
    chunk_size=5_000_000 # Настрой под свой объем ОЗУ
)

Строим модель популярности (с учетом просмотров, батчами)...
  Отфильтровано заказов за период 2025-07-02 - 2025-07-15: 484,645
  Уникальных купленных товаров в периоде: 279,446
  Обрабатываем tracker_df по частям размером ~5,000,000 записей...
    Обработано 1/24 частей tracker_df
    Обработано 11/24 частей tracker_df
    Обработано 21/24 частей tracker_df
    Обработано 24/24 частей tracker_df
  Всего уникальных просмотренных товаров в периоде: 2,951,075
  Рассчитываем комбинированный рейтинг...
  Уникальных товаров с комбинированным рейтингом: 2,951,971
  Выбрано топ-100 популярных товаров (покупки+просмотры)

Топ-10 самых популярных товаров (покупки*3 + просмотры*1):
  1. Товар 12904245: 15677.0 баллов
  2. Товар 51974017: 12288.0 баллов
  3. Товар 207631139: 11808.0 баллов
  4. Товар 172018601: 11406.0 баллов
  5. Товар 26556597: 11070.0 баллов
  6. Товар 113070693: 10743.0 баллов
  7. Товар 165938954: 10628.0 баллов
  8. Товар 11083343: 10547.0 баллов
  9. Товар 219236221: 10353

In [15]:
import pandas as pd
from collections import defaultdict, deque # Добавлен deque
import gc
import tqdm


# --- БЛОК 1: Предпочтения по покупкам (user_purchased_items) ---
print("\n--- БЛОК 1: Предпочтения по покупкам ---")
delivered_orders = orders_df[orders_df['last_status'] == 'delivered_orders']
print(f"  Всего доставленных заказов: {len(delivered_orders):,}")
# Оптимизация: сразу создаем словарь, значения - множества для быстрого добавления уникальных item_id
# Если порядок важен, можно оставить list и преобразовать потом в set при объединении
user_purchased_items = defaultdict(set)
for user_id, item_id in zip(delivered_orders['user_id'], delivered_orders['item_id']):
     user_purchased_items[user_id].add(item_id)

# Преобразуем множества обратно в списки, если это ожидаемый формат
user_purchased_items = {k: list(v) for k, v in user_purchased_items.items()}

print(f"  Пользователей с покупками: {len(user_purchased_items):,}")
del delivered_orders
gc.collect()
print("✅ БЛОК 1 завершен. user_purchased_items создан.")



--- БЛОК 1: Предпочтения по покупкам ---
  Всего доставленных заказов: 10,420,894
  Пользователей с покупками: 805,981
✅ БЛОК 1 завершен. user_purchased_items создан.


In [16]:
print("\n--- БЛОК 2: Предпочтения по просмотрам (ОПТИМИЗИРОВАН) ---")

chunk_size = 5_000_000 # Попробуй уменьшить, если все равно не хватает памяти, например 2_000_000
print(f"  Собираем последние 50 просмотров для тестовых пользователей из tracker_df (батчами по ~{chunk_size:,})...")

# Используем словарь, где ключ - user_id, значение - deque с maxlen=50 для хранения item_id
# deque будет хранить item_id. append() добавляет в конец, maxlen=50 удаляет из начала.
user_viewed_items_deque = defaultdict(lambda: deque(maxlen=50))
test_users_set = set(test_users) # Для быстрого поиска

total_rows = len(tracker_df)
num_chunks = (total_rows // chunk_size) + 1
print(f"  Всего записей в tracker_df: {total_rows:,}. Количество частей: {num_chunks}")

# Обернем цикл в tqdm для визуализации прогресса
for i in tqdm.tqdm(range(num_chunks), desc="Обработка частей tracker_df для просмотров"):
    start_row = i * chunk_size
    end_row = min((i + 1) * chunk_size, total_rows)
    chunk = tracker_df.iloc[start_row:end_row]

    # Фильтрация чанка: только просмотры и только тестовые пользователи
    # Предполагается, что порядок строк в tracker_df примерно соответствует хронологии.
    # Если это не так, этот метод может быть неточным.
    filtered_chunk = chunk[
        (chunk['action_type'] == 'page_view') &
        (chunk['user_id'].isin(test_users_set))
    ]['item_id'] # Берем только item_id, порядок строк сохраняется

    # Накапливаем просмотры, используя append для deque
    # Это эффективно и автоматически ограничивает размер.
    for user_id, item_id in zip(chunk.loc[filtered_chunk.index, 'user_id'], filtered_chunk):
        # Проверка принадлежности user_id к тестовой выборке уже сделана фильтром,
        # но повторная проверка не повредит и может немного ускорить append
        if user_id in test_users_set:
            user_viewed_items_deque[user_id].append(item_id) # Добавляем только item_id

    del chunk, filtered_chunk
    gc.collect()

# --- Преобразование deque в list ---
print("  Преобразуем deque в списки...")
user_viewed_items = {user_id: list(deq) for user_id, deq in user_viewed_items_deque.items()}
# Очищаем промежуточную структуру
del user_viewed_items_deque
gc.collect()

print(f"  Пользователей с просмотрами: {len(user_viewed_items):,}")
# Проверим пример
if user_viewed_items:
    example_user = next(iter(user_viewed_items)) # Получаем первый ключ
    print(f"  Пример для user_id {example_user}: {len(user_viewed_items[example_user])} последних просмотров.")

print("✅ БЛОК 2 (ОПТИМИЗИРОВАН) завершен. user_viewed_items создан.")


--- БЛОК 2: Предпочтения по просмотрам (ОПТИМИЗИРОВАН) ---
  Собираем последние 50 просмотров для тестовых пользователей из tracker_df (батчами по ~5,000,000)...
  Всего записей в tracker_df: 116,552,409. Количество частей: 24


Обработка частей tracker_df для просмотров: 100%|██████████| 24/24 [04:09<00:00, 10.42s/it]


  Преобразуем deque в списки...
  Пользователей с просмотрами: 449,757
  Пример для user_id 4156801: 50 последних просмотров.
✅ БЛОК 2 (ОПТИМИЗИРОВАН) завершен. user_viewed_items создан.


In [17]:
print("\n--- БЛОК 3: Объединение предпочтений ---")
print("  Объединяем покупки и просмотры (ограничиваем 100)...")

# Создаем финальный словарь с ограничением 100 items на пользователя
user_preferences = {}

# Используем test_users для итерации, чтобы покрыть всех
# tqdm для визуализации прогресса объединения
for user_id in tqdm.tqdm(test_users, desc="Объединение предпочтений"):
    prefs_set = set()
    # Добавляем купленные
    prefs_set.update(user_purchased_items.get(user_id, []))
    # Добавляем просмотренные
    prefs_set.update(user_viewed_items.get(user_id, []))

    # Преобразуем в список и ограничиваем 100
    prefs_list = list(prefs_set)
    # Если нужно определенное ранжирование (например, сначала покупки), логику можно усложнить.
    # Сейчас просто берем первые 100 уникальных.
    user_preferences[user_id] = prefs_list[:100]

# Подсчет пользователей с историей
users_with_history = sum(1 for prefs in user_preferences.values() if prefs)

print(f"  Построены предпочтения для {len(user_preferences):,} тестовых пользователей")
print(f"  Пользователей с какой-либо историей (покупки/просмотры): {users_with_history:,}")

# Очистка промежуточных словарей
del user_purchased_items, user_viewed_items
gc.collect()
print("✅ БЛОК 3 завершен. user_preferences создан.")
# --- Конец БЛОК 3 ---

# --- Проверка результата ---
print(f"\n📊 Итог: user_preferences содержит данные для {len(user_preferences)} пользователей.")


--- БЛОК 3: Объединение предпочтений ---
  Объединяем покупки и просмотры (ограничиваем 100)...


Объединение предпочтений: 100%|██████████| 470347/470347 [00:24<00:00, 19359.36it/s]


  Построены предпочтения для 470,347 тестовых пользователей
  Пользователей с какой-либо историей (покупки/просмотры): 466,503
✅ БЛОК 3 завершен. user_preferences создан.

📊 Итог: user_preferences содержит данные для 470347 пользователей.


In [21]:
# === Шаг 6: Генерация рекомендаций (ТОП-100, без купленных/просмотренных товаров) ===
def generate_recommendations(test_users, popular_items, user_preferences):
    """
    Генерирует топ-100 рекомендаций для каждого тестового пользователя,
    исключая товары из его расширенных предпочтений (покупки + последние просмотры).
    """
    print("\n--- Шаг 6: Генерация рекомендаций ---")
    recs = {}
    # Используем tqdm для отслеживания прогресса
    for uid in tqdm.tqdm(test_users, desc='Генерация рекомендаций'):
        # Получаем множество уже купленных/просмотренных (ограничено 100)
        bought_or_viewed = set(user_preferences.get(uid, []))
        # Фильтруем популярные товары, исключая уже взаимодействованные
        # и берем первые 100
        recs[uid] = [item for item in popular_items if item not in bought_or_viewed][:100]
    print(f"  Сгенерированы рекомендации для {len(recs):,} пользователей")
    return recs

# Вызов функции генерации рекомендаций
recommendations = generate_recommendations(test_users, popular_items, user_preferences)


--- Шаг 6: Генерация рекомендаций ---


Генерация рекомендаций: 100%|██████████| 470347/470347 [00:19<00:00, 24223.46it/s]


  Сгенерированы рекомендации для 470,347 пользователей


In [22]:
# === Шаг 7: Формирование submission-файла (готово к сабмиту) ===
def save_submission(recommendations, filename='submission.csv'):
    """
    Сохраняет рекомендации в формате CSV для сабмита.
    """
    print("\n--- Шаг 7: Формирование submission-файла ---")
    rows = []
    # Используем tqdm для отслеживания прогресса
    for uid, items in tqdm.tqdm(recommendations.items(), desc='Формирование submission', total=len(recommendations)):
        # Формируем строку с предсказаниями, разделенными пробелом
        predictions_str = ' '.join(map(str, items))
        rows.append({
            'user_id': uid,
            'item_id_1 item_id_2 ... item_id_100': predictions_str
        })
    # Создаем DataFrame и сохраняем в CSV
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"  ✅ Готово: {filename} (Строк: {len(df):,})")

# Вызов функции сохранения submission-файла
save_submission(recommendations, filename='ozon_baseline_analytic_submission.csv')



--- Шаг 7: Формирование submission-файла ---


Формирование submission: 100%|██████████| 470347/470347 [00:09<00:00, 49768.72it/s]


  ✅ Готово: ozon_baseline_analytic_submission.csv (Строк: 470,347)
